# TILDA Texture Classification

Single notebook pipeline for the MODIA ML 2026 Kaggle challenge.

Constraints used here:
- no pretrained models;
- no transfer learning;
- grayscale TIFF images;
- internal labels are `0..7`, Kaggle submission labels are `1..8`.


## 1. Setup

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
FIGURE_DIR = OUTPUT_DIR / 'figures'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

## 2. Load Metadata

Expected data layout:

```text
data/train.csv
data/train/*.tif
data/test/*.tif
```

In [ ]:
train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir = DATA_DIR / 'test'

df = pd.read_csv(train_csv, sep=';')
df['id'] = df['id'].astype(int)
df['label'] = df['label'].astype(int)
df['path'] = df['id'].map(lambda x: train_dir / f'{x}.tif')

print(df.head())
print(df['label'].value_counts().sort_index())
print('Train images:', len(df))
print('Test images:', len(list(test_dir.glob('*.tif'))))

## 3. Quick Visual Check

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, label in zip(axes.ravel(), sorted(df['label'].unique())):
    row = df[df['label'] == label].sample(1, random_state=SEED).iloc[0]
    img = Image.open(row['path']).convert('L')
    ax.imshow(img, cmap='gray')
    ax.set_title(f'label {label} -> Kaggle {label + 1}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'samples_by_class.png', dpi=150)
plt.show()

## 4. Split + Transforms

Augmentation is applied only to the training split. Validation/test use deterministic preprocessing.

In [ ]:
IMG_SIZE = (384, 256)  # height, width; keeps the original 3:2 ratio after resizing
BATCH_SIZE = 64
NUM_WORKERS = 4

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label'],
)

train_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05), shear=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.15, scale=(0.01, 0.06), ratio=(0.3, 3.3)),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

len(train_df), len(val_df)

## 5. Dataset + DataLoaders

In [ ]:
class TildaDataset(Dataset):
    def __init__(self, dataframe=None, image_dir=None, ids=None, transform=None, has_labels=True):
        self.dataframe = dataframe.reset_index(drop=True) if dataframe is not None else None
        self.image_dir = Path(image_dir) if image_dir is not None else None
        self.ids = list(ids) if ids is not None else None
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.dataframe) if self.has_labels else len(self.ids)

    def __getitem__(self, idx):
        if self.has_labels:
            row = self.dataframe.iloc[idx]
            image_id = int(row['id'])
            path = Path(row['path'])
            label = int(row['label'])
        else:
            image_id = int(self.ids[idx])
            path = self.image_dir / f'{image_id}.tif'
            label = -1

        image = Image.open(path).convert('L')
        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_id


train_ds = TildaDataset(train_df, transform=train_tfms, has_labels=True)
val_ds = TildaDataset(val_df, transform=eval_tfms, has_labels=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

xb, yb, ids = next(iter(train_loader))
xb.shape, yb[:8]

## 6. Models From Scratch

Start with a strong but not oversized custom CNN. It is usually a better first baseline than a huge network from scratch on only 2361 images.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)


class TextureCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32, dropout=0.05),
            ConvBlock(32, 64, dropout=0.05),
            ConvBlock(64, 128, dropout=0.10),
            ConvBlock(128, 256, dropout=0.10),
            ConvBlock(256, 384, dropout=0.15),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(384, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = TextureCNN(num_classes=8).to(DEVICE)
sum(p.numel() for p in model.parameters())

## 7. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for images, labels, _ in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def train_model(model, train_loader, val_loader, epochs=80, lr=3e-4, weight_decay=1e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []
    best_acc = -1.0
    best_path = CHECKPOINT_DIR / 'best_texture_cnn.pt'
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'lr': scheduler.get_last_lr()[0],
        }
        history.append(row)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({'model_state_dict': model.state_dict(), 'history': history, 'best_acc': best_acc}, best_path)

        print(f"Epoch {epoch:03d} | train {train_acc:.4f}/{train_loss:.4f} | val {val_acc:.4f}/{val_loss:.4f} | best {best_acc:.4f}")

    print(f'Training time: {(time.time() - start) / 60:.1f} min')
    return pd.DataFrame(history), best_path


## 8. Train

For a quick local test, set `EPOCHS = 2`. On EDiTO/A6000, start with `EPOCHS = 80`.

In [ ]:
EPOCHS = 80
history, best_path = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=3e-4, weight_decay=1e-4)
history.tail()

## 9. Validation Diagnostics

In [ ]:
history.plot(x='epoch', y=['train_acc', 'val_acc'], figsize=(8, 4), grid=True)
plt.savefig(FIGURE_DIR / 'accuracy_curve.png', dpi=150)
plt.show()

history.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(8, 4), grid=True)
plt.savefig(FIGURE_DIR / 'loss_curve.png', dpi=150)
plt.show()

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_preds, val_labels = [], []
with torch.no_grad():
    for images, labels, _ in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        val_preds.extend(logits.argmax(dim=1).cpu().numpy())
        val_labels.extend(labels.numpy())

print('Best validation accuracy:', accuracy_score(val_labels, val_preds))
cm = confusion_matrix(val_labels, val_preds, labels=list(range(8)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[str(i + 1) for i in range(8)])
fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Validation confusion matrix, Kaggle labels 1..8')
plt.savefig(FIGURE_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## 10. Predict Test + Create Submission

Kaggle expects labels `1..8`, while the model predicts `0..7`, so we save `prediction + 1`.

In [ ]:
test_ids = sorted([int(p.stem) for p in test_dir.glob('*.tif')])
test_ds = TildaDataset(image_dir=test_dir, ids=test_ids, transform=eval_tfms, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

test_predictions = []
model.eval()
with torch.no_grad():
    for images, _, image_ids in tqdm(test_loader):
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy() + 1
        for image_id, pred in zip(image_ids.numpy(), preds):
            test_predictions.append({'id': int(image_id), 'label': int(pred)})

submission = pd.DataFrame(test_predictions).sort_values('id')
submission_path = SUBMISSION_DIR / 'submission_texture_cnn.csv'
submission.to_csv(submission_path, index=False)
submission.head(), submission_path

## 11. Next Experiments

After the first Kaggle submission:

1. Try `IMG_SIZE = (512, 384)` if memory allows.
2. Add a ResNet18-from-scratch model for comparison.
3. Use 5-fold stratified training and average fold predictions.
4. Add test-time augmentation (horizontal/vertical flips) only if validation improves.